# 🎙️ Fine-Tuning SOTA Multilingual XLS-R for Speech Emotion Recognition on Google GPU/TPU

This notebook enables you to fine-tune **Meta's XLS-R (Wav2Vec2)** speech transformer model on Speech Emotion Recognition (SER) datasets. You can train it on **Hindi, Telugu, or English (RAVDESS)** speech.

Once trained, the notebook automatically packages and zips the model so you can download it and run it inside your local dashboard.

### ⚡ Hardware Accelerator Required
Before running, go to **Runtime > Change runtime type** in the top menu and select a GPU (e.g., **T4 GPU**, **L4 GPU**, or **A100**) or **TPU v5e**. This notebook automatically detects the device and configures the optimal precision (`fp16` or `bf16`) for efficient training.

In [ ]:
# 1. Install required libraries
!pip install transformers datasets evaluate accelerate soundfile librosa openpyxl

In [ ]:
# 2. Import core libraries and configure the device
import os
import sys
import torch
import librosa
import numpy as np
from datasets import Dataset, DatasetDict
from transformers import Wav2Vec2FeatureExtractor, Wav2Vec2ForSequenceClassification, TrainingArguments, Trainer
import evaluate

has_tpu = False
try:
    # Check if running on Google TPU
    import torch_xla.core.xla_model as xm
    device = xm.xla_device()
    has_tpu = True
    print(f"Using Google TPU Core: {device}")
except ImportError:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"TPU libraries not found. Using device: {device}")

### 📁 Data Preparation
To train the model, you need audio clips (.wav) and their corresponding emotion labels. Underneath is a helper function to set up your dataset. You can easily upload a ZIP folder containing your `.wav` files to Colab.

In [ ]:
# 3. Programmatic Loader for RAVDESS (Example)
# If you want to train on English RAVDESS first:
!wget -O Audio_Speech_Actors_01-24.zip https://zenodo.org/api/records/1188976/files/Audio_Speech_Actors_01-24.zip/content
!unzip -q Audio_Speech_Actors_01-24.zip -d ravdess_data

In [ ]:
# 4. Prepare Dataset for Hugging Face
import glob

EMOTIONS = ["neutral", "calm", "happy", "sad", "angry", "fearful", "disgust", "surprised"]
EMOTION_MAP = {f"{i+1:02d}": i for i in range(len(EMOTIONS))}

def build_dataset_dict(audio_dir):
    wav_files = glob.glob(os.path.join(audio_dir, "**", "*.wav"), recursive=True)
    
    data = {"path": [], "label": []}
    for f in wav_files:
        filename = os.path.basename(f)
        parts = filename.split('-')
        if len(parts) >= 3:
            emotion_code = parts[2]
            if emotion_code in EMOTION_MAP:
                data["path"].append(f)
                data["label"].append(EMOTION_MAP[emotion_code])
                
    hf_dataset = Dataset.from_dict(data)
    # Split 80/20 train/val
    split_ds = hf_dataset.train_test_split(test_size=0.2, seed=42)
    return DatasetDict({"train": split_ds["train"], "validation": split_ds["test"]})

dataset = build_dataset_dict("ravdess_data")
print(dataset)

### 🏷️ Feature Extraction
We use the pre-trained `Wav2Vec2FeatureExtractor` to resample and normalize inputs to 16,000 Hz.

In [ ]:
# 5. Preprocess audio paths to raw arrays
model_checkpoint = "facebook/wav2vec2-xls-r-300m"
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(model_checkpoint)

def preprocess_audio(batch):
    audio_arrays = []
    for path in batch["path"]:
        # Load and automatically resample to 16kHz
        speech, _ = librosa.load(path, sr=16000)
        audio_arrays.append(speech)
        
    inputs = feature_extractor(
        audio_arrays, 
        sampling_rate=16000, 
        max_length=16000 * 3, # 3 seconds max pad
        truncation=True, 
        padding="max_length"
    )
    batch["input_values"] = inputs.input_values
    return batch

# Map preprocessor (using batch mode for speed)
encoded_dataset = dataset.map(preprocess_audio, batched=True, batch_size=16, remove_columns=["path"])

### 🚀 Model Training Setup

In [ ]:
# 6. Load Pre-trained Transformer Model with Explicit Labels
id2label = {str(i): label for i, label in enumerate(EMOTIONS)}
label2id = {label: i for i, label in enumerate(EMOTIONS)}

model = Wav2Vec2ForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(EMOTIONS),
    id2label=id2label,
    label2id=label2id,
    classifier_proj_size=256
)

# Freezing the base CNN feature extractor layers to prevent overfitting
model.freeze_feature_encoder()

# Freeze first 12 encoder blocks of the 24 transformer layers to prevent overfitting
for layer in model.wav2vec2.encoder.layers[:12]:
    for param in layer.parameters():
        param.requires_grad = False

model.to(device)

In [ ]:
# 7. Define accuracy metrics
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions = np.argmax(eval_pred.predictions, axis=1)
    return metric.compute(predictions=predictions, references=eval_pred.label_ids)

In [ ]:
# 8. Configure Trainer and Training Arguments
use_bf16 = False
use_fp16 = False

if has_tpu:
    use_bf16 = True
elif torch.cuda.is_available():
    # Check if GPU supports bf16 (Ampere architectures like A100/L4, compute capability >= 8)
    if torch.cuda.get_device_properties(0).major >= 8:
        use_bf16 = True
    else:
        use_fp16 = True

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,          # Upgraded learning rate for fine-tuning
    lr_scheduler_type="linear",  # Upgraded: linear decay
    warmup_ratio=0.1,            # Upgraded: warmup scheduler
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=15,         # Upgraded: 15 epochs for convergence
    weight_decay=0.01,
    metric_for_best_model="accuracy",
    load_best_model_at_end=True,
    logging_steps=10,
    bf16=use_bf16,               # Auto-configured precision
    fp16=use_fp16,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["validation"],
    processing_class=feature_extractor,
    compute_metrics=compute_metrics
)

In [ ]:
# 9. Run the Training loop
trainer.train()

### 💾 Save and Export Model

In [ ]:
# 10. Save models locally and download zip
os.makedirs("xlsr-ser", exist_ok=True)
trainer.save_model("./xlsr-ser")
feature_extractor.save_pretrained("./xlsr-ser")

print("[SUCCESS] Model saved successfully! Packaging ZIP...")
!zip -r xlsr-ser.zip ./xlsr-ser
print("[INFO] xlsr-ser.zip is ready for download in your file browser (left panel). Upload it back to your local models folder.")